This Jupyter Notebook contains the training and evaluation.
See the Jupyter Notebook "1_Load_data","2_Exploring the data" and "3_Preprocessing" for the previous steps.
Finally the analysis of the data is in the third Jupter Notebook, "5_Visualisation".

In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import pickle as pkl
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np

In [ ]:
#connect to google drive - the file 'k_fold_data' should be uploaded
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# training on cpu very slow, if possible use gpu, still even with T4 it will take at least 2-3 hours per Epoch, which in my case are 2 epochs for each fold, 10 epochs ~ 30 hours of computational time
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
os.environ['TOKENIZERS_PARALLELISM'] = 'false' # there might be interferences with the parallelism of the Hugging Face Trainer
os.environ['WANDB_DISABLED'] = "true"

print(f"Using device: {device}")
if torch.device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

Using device: cuda:0


In [ ]:
#file names
k_fold_test_file_name='/content/drive/MyDrive/data/k_fold_data_sicherheits_kopie.xlsx'
model_name=''
full_data_name=''
independent_file_name=''
independent_data_name=''

In [ ]:
#read data into pandas data frame
data = k_fold_test_file_name # change to 'full_data_name' for evaluation, 'independent_file_name', 'independent_data_name'
df = pd.read_excel(data, index_col=0)
df.shape

(88219, 10)

In [ ]:
# set seeds
torch.manual_seed(123)
np.random.seed(123)

In [ ]:
# Set up parameters
bert_model_name= 'bert-base-german-cased'
num_classes = 9
max_length = 256
batch_size = 20
num_epochs = 2
learning_rate = 2e-5
#output_dir ='data/run'+ str(1)

In [ ]:
#quick check to see all labels there (0-8)
print("Unique labels/partys in dataset:{}".format(df['party'].unique()))

Unique labels/partys in dataset:[0 1 2 3 4 5 6 7 8]


In [ ]:
"""takes a text and returns a list of texts in given length"""
def chunksspeech(text,term, length):
    return [' '.join(chunk) for chunk in list((text[0+i:length+i] for i in range(0, int(term), length)))]

In [ ]:
'''chunks the speeches and creates a map to link each speech-chunk to the corresponding party label for training'''
def chunk_data(texts,labels,speech_ids,length):
    chunked_speeches=[chunksspeech(text.split(" "),len(text.split(" ")),length) for text in texts]
    speech_chunks = [chunk for speech in chunked_speeches for chunk in speech]
    chunk_labels = [label for label,speech in zip(labels,chunked_speeches) for _ in speech]
    chunk_to_speech_mapping = [speech_id for speech_id,speech in zip(speech_ids,chunked_speeches) for _ in speech]
    return speech_chunks,chunk_labels,chunk_to_speech_mapping

In [ ]:
'''Creates the data set to be fed into the model
takes list of text, labels, speech-ids, tokenizer and maximum length
returns tokenized text (with maximum length padding) with input_id,attention_mask,label and speech_id  '''
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, ids, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.ids = ids
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        speech_id = self.ids[idx]
        encoding = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(label,dtype=torch.long), 'speech_id': speech_id}

In [ ]:
'''
BERT Classifier with a BERT layer, Dropout layer and a linear layer
'''
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output #should be size batch_size/hidden_size
        x = self.dropout(pooled_output)
        logits = self.fc(x)
        return logits

In [ ]:
def train(model, data_loader, optimizer, scheduler, device):
    model.train()
    cross_entropy_loss = 0
    for batch in tqdm(data_loader,desc="Training"):
        # Reset gradients before first run - if training
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        # forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        # backpropagation with optimizer step
        loss.backward()
        cross_entropy_loss += loss.item()
        optimizer.step()
        scheduler.step()
    cross_entropy_loss = cross_entropy_loss/len(data_loader)
    return cross_entropy_loss

In [ ]:
def evaluate(model, data_loader, device, ids=None):
    model.eval()
    predictions = []
    actual_labels = []
    all_probs = []
    all_ids = []  # collect speech Id to aggregate later
    with torch.no_grad():
        for batch in tqdm(data_loader,desc="Evaluation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            probs = nn.functional.softmax(outputs, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
            all_ids.extend(batch['speech_id'])  # edit to process chunks#'speech_id'
    return accuracy_score(actual_labels, predictions),classification_report(actual_labels, predictions), all_probs, all_ids #classification_report(actual_labels, predictions)

In [ ]:
"""Creates a dataframe with speech_id and predicitions on Chunk-level and aggregates to speech-level """

def speech_chunk_dataframe(prob, speech_ids):
    prob_df = pd.DataFrame(prob)
    speech_ids = [x.item() for x in speech_ids]
    prob_df['speechnumber'] = speech_ids
    return prob_df

"Group chunk-level probabilites by speech_id and compute weighted mean if provided"

def group_probablities_by_speech(probabilities_df, weights=None):
  grouped_probabilities = probabilities_df.groupby('speechnumber').mean()
  return grouped_probabilities

'''Evaluate on speech level'''

def evaluate_speeches(probabilities_df, speech_id_to_party_map, weights=None):
    grouped_probabilities = group_probablities_by_speech(probabilities_df, weights)
    # get highest average probability per speech
    predicted_speech_labels = grouped_probabilities.idxmax(axis=1).tolist()
    # get true labels
    grouped_speech_ids = grouped_probabilities.index.tolist()
    true_labels= [speech_id_to_party_map[k] for k in grouped_speech_ids]
    # get accuracy and report
    accuracy = accuracy_score(true_labels, predicted_speech_labels)
    report = classification_report(true_labels, predicted_speech_labels)

    return  accuracy, report,true_labels, predicted_speech_labels,grouped_probabilities

In [ ]:
def run_fold_training(fold, df):
    fold= fold+1
    df_train = df[df['k_fold'] != fold]
    df_val = df[df['k_fold'] == fold]

    t_texts=list(df_train['text'])
    t_labels=list(df_train['party'])
    t_mps=list(df_train['speaker'])
    t_speech_ids=list(df_train['speechnumber'])


    v_texts=list(df_val['text'])
    v_labels=list(df_val['party'])
    v_mps=list(df_val['speaker'])
    v_speech_ids=list(df_val['speechnumber'])


    train_texts, train_labels, train_ids = chunk_data(t_texts,t_labels,t_speech_ids,max_length)
    val_texts, val_labels, val_ids = chunk_data(v_texts,v_labels,v_speech_ids,max_length)

    tokenizer = BertTokenizer.from_pretrained(bert_model_name)

    train_dataset = TextClassificationDataset(train_texts, train_labels,train_ids, tokenizer, max_length)
    val_dataset = TextClassificationDataset(val_texts, val_labels,val_ids, tokenizer, max_length)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size)


    model = BERTClassifier(bert_model_name, num_classes).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    total_steps = len(train_dataloader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        cross_entropy_loss = train(model, train_dataloader, optimizer, scheduler, device)
        accuracy, report, probabilities, speech_ids = evaluate(model, val_dataloader, device)
        print(f"Validation Accuracy Chunks: {accuracy:.4f}")
        print(report)
        # Aggregate on speech-level
        speech_level_df = speech_chunk_dataframe(probabilities, speech_ids)
        # Evaluate on speech-level
        speech_accuracy, speech_report, true_speech_labels, predicted_speech_labels, speech_probabilities = evaluate_speeches(speech_level_df, speech_party_map, weights=None)
        print(f"Validation Accuracy Speeches: {speech_accuracy:.4f}")
        print(speech_report)

    model_k_fold_name= ("/content/drive/MyDrive/data/bert_classifier_5_k_fold_{}.pth").format(fold)
    torch.save(model.state_dict(), model_k_fold_name)

In [ ]:
dfs=[]
kfold=5
for j in range(kfold):
    temp_df= run_fold_training(j,df)
    #fin_df = pd.concat(dfs)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/485k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-german-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/2


Training:   0%|          | 0/9710 [00:00<?, ?it/s]